In [1]:
print("Hello world")


Hello world


In [2]:
from pypdf import PdfReader
import tiktoken
import spacy
import json
import re


In [3]:
file_path = "test.pdf"
reader = PdfReader(file_path)


In [4]:
# Load spaCy model for sentence segmentation
nlp = spacy.load("en_core_web_sm")


In [5]:

def clean_text(text):
    # remove bullet artifacts
    text = re.sub(r"[•\u2022]+", " ", text)

    # remove excessive whitespace/newlines
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [6]:
def is_valid_chunk(text):
    if len(text.strip()) < 30:
        return False

    # mostly bullet junk
    if len(re.sub(r"[^a-zA-Z]", "", text)) < 20:
        return False

    return True

In [7]:
def split_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]


In [8]:
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(enc.encode(text))


In [9]:
pages = []

for i, page in enumerate(reader.pages):
    text = page.extract_text()

    if text.strip():
        sentences = split_sentences(text)

        pages.append({
            "page": i + 1,
            "sentences": sentences
        })


In [10]:
print(pages)

[{'page': 2, 'sentences': ['Introduction to Data Science\nWhat is Data Science?', 'Welcome to this course on Data Science.', 'If you are active on Fb/Insta/X, share your\nData Science journey on these platforms and tag me.', 'I would love to see your\nprogress.', 'I am so happy you decided to learn Data Science with me in my style.', 'Lets step\nback and talk a bit about why do we need data science.', 'What kind of tasks we will\ndo as a data scientist in an organization\nWhat is Data Science?', 'At its core, Data Science is the field of study that uses mathematics, statistics,\nprogramming, and domain knowledge to extract meaningful insights from data.', 'Well that sounds like a bookish definition which you are free to write in your\nsemester exams but Data Science pretty much blends various techniques from\ndifferent disciplines to analyze large amounts of information and solve real-world\nproblems.', 'Simple Definition:\nLet me give you a very simple non fancy definition of Data Sci

In [11]:
def chunk_document(pages, chunk_size=250):
    all_chunks = []
    chunk_id = 0

    for page in pages:
        sentences = page["sentences"]
        i = 0

        while i < len(sentences):
            current_chunk = []
            current_tokens = 0
            chunk_pages = set()

            start_idx = i

            while i < len(sentences):
                sent = sentences[i]
                sent_tokens = count_tokens(sent)

                if current_tokens + sent_tokens > chunk_size:
                    break

                current_chunk.append(sent)
                current_tokens += sent_tokens
                chunk_pages.add(page["page"])
                i += 1

            # handle edge case: single long sentence
            if not current_chunk and i < len(sentences):
                current_chunk.append(sentences[i])
                current_tokens = count_tokens(sentences[i])
                chunk_pages.add(page["page"])
                i += 1

            chunk_text = clean_text(" ".join(current_chunk))

            if is_valid_chunk(chunk_text):
                all_chunks.append({
                    "chunk_id": chunk_id,
                    "pages": sorted(list(chunk_pages)),
                    "text": chunk_text,
                    "tokens": current_tokens,
                    "num_sentences": len(current_chunk)
                })

                chunk_id += 1

            # overlap control (simple + safe)
            i = max(start_idx + 1, i - 1)

    return all_chunks

In [12]:
chunks = chunk_document(pages)


In [13]:
with open('chunks.json', 'w') as file:
    json.dump(chunks, file, indent=4, sort_keys=True) # Adds whitespace and sorts keys
